In [1]:
print(123)

123


In [2]:
# Comparing 2 queries against a document: 

from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-07-14 19:46:59.328718244 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
# compute similarities
v1.dot(dv)

np.float64(0.3233238425677314)

In [4]:
# second similarity 
v2.dot(dv)

np.float64(0.01973045838992233)

In [6]:
# fetch ingest.py (faq dataset)
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-07-14 19:48:02--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-14 19:48:02 (41.2 MB/s) - ‘ingest.py’ saved [738/738]



In [ ]:
# load the documents
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
# combine question and answer for each document 
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [9]:
# embed in batches 

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/28 [00:00<?, ?it/s]

In [10]:
# search 

query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [ ]:
"""
AVAILABLE MODELS: 

Xenova/all-MiniLM-L6-v2 (384d) - best small general-purpose
Xenova/all-MiniLM-L12-v2 (384d) - better quality, slower
Xenova/paraphrase-MiniLM-L6-v2 (384d) - paraphrase detection
Xenova/paraphrase-multilingual-MiniLM-L12-v2 (384d) - multilingual
Xenova/multilingual-e5-small (384d) - multilingual retrieval
Xenova/multilingual-e5-base (768d) - stronger multilingual
Xenova/bge-small-en-v1.5 (384d) - strong retrieval
Xenova/bge-base-en-v1.5 (768d) - stronger retrieval
Xenova/gte-small (384d) - lightweight modern model
Xenova/gte-base (768d) - stronger GTE

o use a different model, add it to download.py, run the download, then update the path:

embed = Embedder("models/Xenova/bge-base-en-v1.5")
vectors = embed.encode("your text here")
print(vectors.shape)



--> THIS CAN BE DEPLOYED IN MINIMAL ENVIRONMENTS SUCH AS 
- SMALL DOCKER IMAGES 
SERVERLESS FUNCTIONS 
EDGE DEVICES 
"""